In [2]:
# ── LSTM.ipynb — imports, reproducibility, device ────────────────────────
import numpy as np
import pandas as pd
import torch
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

import features as F

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)          # 'mps' on Apple Silicon — much faster than cpu

SEQ_LEN  = 55
VAL_FRAC = 0.2

device: mps


In [4]:
import importlib, features as F
importlib.reload(F)
print([f for f in dir(F) if f.startswith("build")])   # should list build_features_gbdt

['build_features', 'build_features_gbdt']


In [5]:
# 1. LGB val preds WITH temporal features (on your val split)
import features as F
import lightgbm as lgb

rich = F.build_features_gbdt(pd.read_csv("Data/train.csv"))
gcols = F.feature_columns(rich); cols = gcols + ["stock_id"]
dates = np.sort(rich.date_id.unique()); cut = dates[-int(len(dates)*0.2)]
tr, va = rich[rich.date_id < cut], rich[rich.date_id >= cut]
Xtr = tr[cols].replace([np.inf,-np.inf],np.nan).fillna(0); ytr=tr.target; ok=ytr.notna().values
m = lgb.LGBMRegressor(objective="mae",n_estimators=500,learning_rate=0.03,num_leaves=31,
    min_child_samples=1000,reg_alpha=1.0,reg_lambda=1.0,subsample=0.7,subsample_freq=1,
    colsample_bytree=0.7,random_state=42,n_jobs=-1,verbose=-1).fit(Xtr[ok],ytr[ok],categorical_feature=["stock_id"])
va[["row_id"]].assign(lgb=m.predict(va[cols].replace([np.inf,-np.inf],np.nan).fillna(0))).to_parquet("preds_lgb_val.parquet", index=False)

In [ ]:
# 2. 3-model blend: merge, search weights, zero-sum
g = pd.read_parquet("preds_gru_val.parquet"); l = pd.read_parquet("preds_lstm_val.parquet")
b = pd.read_parquet("preds_lgb_val.parquet")
m = g.merge(l[["row_id","lstm"]] if "lstm" in l else l.rename(columns={l.columns[-1]:"lstm"}), on="row_id") \
     .merge(b, on="row_id").dropna(subset=["target"])
zero = m["target"].abs().mean()
def sk(p): return (zero - (m["target"]-p).abs().mean())/zero*100
print("corr gru/lstm:", round(m.gru.corr(m.lstm),3), "| gru/lgb:", round(m.gru.corr(m.lgb),3))

best = (0,None)
for wl in np.arange(0.3,0.71,0.1):                     # lgb weight
    for wg in np.arange(0.0,1.0-wl+0.01,0.1):          # gru; lstm = rest
        ws = 1-wl-wg
        if ws < 0: continue
        blend = wl*m.lgb + wg*m.gru + ws*m.lstm
        blend = blend - blend.groupby([m.date_id, m.seconds_in_bucket]).transform("mean")
        s = sk(blend)
        if s > best[0]: best = (s, (round(wl,1),round(wg,1),round(ws,1)))
print(f"best 3-model skill {best[0]:.2f}%  weights (lgb,gru,lstm)={best[1]}")

corr gru/lstm: 0.957 | gru/lgb: 0.855
best 3-model skill 2.22%  weights (lgb,gru,lstm)=(np.float64(0.5), np.float64(0.5), np.float64(0.0))


In [7]:
# 2. 3-model blend but with transformer instead of lstm: merge, search weights, zero-sum
g = pd.read_parquet("preds_gru_val.parquet"); l = pd.read_parquet("preds_transformer_val.parquet")
b = pd.read_parquet("preds_lgb_val.parquet")
m = g.merge(l[["row_id","transformer"]] if "transformer" in l else l.rename(columns={l.columns[-1]:"transformer"}), on="row_id") \
     .merge(b, on="row_id").dropna(subset=["target"])
zero = m["target"].abs().mean()
def sk(p): return (zero - (m["target"]-p).abs().mean())/zero*100
print("corr gru/transformer:", round(m.gru.corr(m.transformer),3), "| gru/lgb:", round(m.gru.corr(m.lgb),3))
best = (0,None)
for wl in np.arange(0.3,0.71,0.1):                     # lgb weight
    for wg in np.arange(0.0,1.0-wl+0.01,0.1):          # gru; lstm = rest
        ws = 1-wl-wg
        if ws < 0: continue
        blend = wl*m.lgb + wg*m.gru + ws*m.transformer
        blend = blend - blend.groupby([m.date_id, m.seconds_in_bucket]).transform("mean")
        s = sk(blend)
        if s > best[0]: best = (s, (round(wl,1),round(wg,1),round(ws,1)))
print(f"best 3-model skill {best[0]:.2f}%  weights (lgb,gru,transformer)={best[1]}")

corr gru/transformer: 0.892 | gru/lgb: 0.855
best 3-model skill 2.40%  weights (lgb,gru,transformer)=(np.float64(0.3), np.float64(0.1), np.float64(0.6))
